# Parameter Golf - Google Colab

OpenAI Model Craft Challenge: Parameter Golf の検証用ノートブック。

**ランタイム設定**: 上メニューから「ランタイム」→「ランタイムのタイプを変更」→ **T4 GPU** を選択してください。

## 1. GPU確認

In [ ]:
!nvidia-smi

## 2. リポジトリのクローンと依存関係インストール

In [ ]:
!git clone https://github.com/tsubasagit/parameter-golf.git /content/parameter-golf
%cd /content/parameter-golf
!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

## 3. データセットのダウンロード（最小構成: 1 shard）

In [ ]:
!python3 data/cached_challenge_fineweb.py --variant sp1024 --train-shards 1

---
## 4. ベースライン実行

まずオリジナルの `train_gpt.py` でベースラインスコアを取得します。

T4 (16GB VRAM) 向けにバッチサイズを縮小。

In [ ]:
import os
os.environ['RUN_ID'] = 'colab_baseline'
os.environ['ITERATIONS'] = '500'
os.environ['TRAIN_BATCH_TOKENS'] = '131072'
os.environ['VAL_LOSS_EVERY'] = '100'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['MAX_WALLCLOCK_SECONDS'] = '600'

!torchrun --standalone --nproc_per_node=1 train_gpt.py

---
## 5. 改良版: 上位テクニック適用

リーダーボード2位（1.1458 BPB）のスクリプトを使用。

### 適用テクニック:
| テクニック | 効果 |
|-----------|------|
| MLP 3x拡張 | 最大の改善要因。hidden dim 1024→1536 |
| SmearGate | 前トークンとの学習可能なゲート融合 |
| BigramHash(4096) | トークンペアのハッシュ埋め込み |
| U-Net Skip | エンコーダ→デコーダのスキップ接続 |
| SWA | 学習後半のチェックポイント平均 |
| Muon WD | Weight Decay追加で量子化品質向上 |
| Sliding Window Eval | stride=64で評価精度向上 |
| seq_len 2048 | コンテキスト長2倍 |

In [ ]:
# 2位エントリのスクリプトをコピーして使用
import shutil
src = '/content/parameter-golf/records/track_10min_16mb/2026-03-20_Int6_MLP3x_SmearGate_BigramHash_MuonWD_SWA/train_gpt.py'
dst = '/content/parameter-golf/train_gpt_improved.py'
shutil.copy2(src, dst)
print(f'Copied improved script to {dst}')

In [ ]:
import os

# --- 基本設定 ---
os.environ['RUN_ID'] = 'colab_improved_v1'
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp1024'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_1024_bpe.model'
os.environ['VOCAB_SIZE'] = '1024'

# --- T4向けに調整 ---
os.environ['ITERATIONS'] = '500'
os.environ['MAX_WALLCLOCK_SECONDS'] = '900'
os.environ['TRAIN_BATCH_TOKENS'] = '65536'     # T4 VRAM向け縮小 (H100: 786432)
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['TRAIN_SEQ_LEN'] = '1024'            # T4向け縮小 (H100: 2048)
os.environ['VAL_LOSS_EVERY'] = '100'

# --- 上位テクニックのハイパラ ---
os.environ['NUM_LAYERS'] = '9'
os.environ['MODEL_DIM'] = '512'
os.environ['NUM_HEADS'] = '8'
os.environ['NUM_KV_HEADS'] = '4'
os.environ['MLP_MULT'] = '3'                    # 3x MLP拡張
os.environ['MATRIX_LR'] = '0.02'
os.environ['SCALAR_LR'] = '0.02'
os.environ['MUON_MOMENTUM'] = '0.99'
os.environ['WARMDOWN_ITERS'] = '200'            # 短縮 (H100: 3000)

!torchrun --standalone --nproc_per_node=1 train_gpt_improved.py

---
## 6. 結果比較

| 実行 | 期待 val_bpb | 備考 |
|------|-------------|------|
| ベースライン (セル4) | ~1.3+ | 9L, MLP2x, 500iter, T4 |
| 改良版 (セル5) | ~1.25前後 | SmearGate+BigramHash+MLP3x, 500iter, T4 |
| SOTA (8xH100, full) | 1.1428 | 全テクニック, 20000iter, 8xH100 |

T4 + 500iterではSOTAには届きませんが、**テクニックの効果を相対比較**できます。

改良版のスコアがベースラインより良ければ、テクニックが有効に機能しています。

---
## 7. カスタム実験

以下のパラメータを変えて再実行し、効果を検証できます。

In [ ]:
import os

os.environ['RUN_ID'] = 'colab_experiment_custom'
os.environ['DATA_PATH'] = './data/datasets/fineweb10B_sp1024'
os.environ['TOKENIZER_PATH'] = './data/tokenizers/fineweb_1024_bpe.model'
os.environ['VOCAB_SIZE'] = '1024'

os.environ['ITERATIONS'] = '500'
os.environ['MAX_WALLCLOCK_SECONDS'] = '900'
os.environ['TRAIN_BATCH_TOKENS'] = '65536'
os.environ['VAL_BATCH_SIZE'] = '65536'
os.environ['VAL_LOSS_EVERY'] = '100'

# --- 実験パラメータ（ここを変更） ---
os.environ['NUM_LAYERS'] = '10'                  # 10層に増やす
os.environ['TRAIN_SEQ_LEN'] = '1024'
os.environ['MLP_MULT'] = '3'
os.environ['MATRIX_LR'] = '0.02'
os.environ['MUON_MOMENTUM'] = '0.99'
os.environ['WARMDOWN_ITERS'] = '200'

# BigramHash: 環境変数で調整可能な場合
# os.environ['BIGRAM_VOCAB_SIZE'] = '10240'      # 4096→10240

!torchrun --standalone --nproc_per_node=1 train_gpt_improved.py